In [33]:
import os
import re
import time
import json
import pickle
import random
import requests
import pandas as pd
from bs4 import BeautifulSoup

# Selenium for web automation
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

# Alternative scraper
import undetected_chromedriver as uc  


In [2]:
russell3000= pd.read_stata(
    r'../../../../../Onedrive/Projects/PossibleProjects/20250520 Union Glassdoor/Russell3000.dta').T.to_dict()
print(len(russell3000))

3861


In [3]:
russell3000[0]

{'gvkey': 'GVKEY_001004',
 'conm': 'AAR CORP',
 'glassdoor_web': 'https://www.glassdoor.com/Reviews/AAR-Reviews-E4.htm',
 'flag': 'sp1500'}

In [50]:
# Get the directory where the script is running (for Jupyter Notebook)
script_dir = os.getcwd()

# Define the folder path in the same directory as the script
folder_path = os.path.join(script_dir, "Review_202502")

# Check if the directory exists, if not, create it
if not os.path.exists(folder_path):
    os.makedirs(folder_path)
    print(f"Created directory: {folder_path}")
else:
    print(f"Directory already exists: {folder_path}")

Created directory: C:\Users\xxiao4\Documents\Amanda RA\Review_202502


In [11]:
options = uc.ChromeOptions()
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/132.0.6834.111 Safari/537.36")
driver = uc.Chrome(options=options)
'''
options = uc.ChromeOptions()
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/132.0.6834.111 Safari/537.36")

driver = uc.Chrome(options=options)
'''
# Step 2: Login and save cookies
def login_and_save_cookies(username, password):
    try:
        # Open the Glassdoor login page
        driver.get("https://www.glassdoor.com/profile/login_input.htm")

        # Enter the email
        email_input = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.ID, "inlineUserEmail"))
        )
        email_input.send_keys(username
                             )

        # Click the Continue button
        continue_button = WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.XPATH, "//button[@data-test='email-form-button']"))
        )
        continue_button.click()

        # Enter the password
        password_input = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.ID, "inlineUserPassword"))
        )
        password_input.send_keys(password)

        # Click the Sign In button
        signin_button = WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.XPATH, "//button[@type='submit' and @data-role-variant='primary']"))
        )
        signin_button.click()

        # Wait for the login process to complete
        WebDriverWait(driver, 10).until(EC.title_contains("Glassdoor"))
        print("Login successful!")

        # Retrieve and save cookies to a file
        cookies = driver.get_cookies()
        with open("glassdoor_cookies.json", "w") as cookie_file:
            json.dump(cookies, cookie_file)

        print("✅ Login successful! Cookies saved.")

    except Exception as e:
        print(f"❌ Error during login: {e}")

# Step 3: Load cookies and access the page
def load_cookies_and_access():
    try:
        # Step 1: Open the target page
        driver.get("https://www.glassdoor.com/profile/login_input.htm")
        time.sleep(3)

        # Step 2: Load cookies from file
        with open("glassdoor_cookies.json", "r") as cookie_file:
            saved_cookies = json.load(cookie_file)

        for cookie in saved_cookies:
            driver.add_cookie(cookie)

        print("✅ Cookies loaded successfully.")

        # Step 3: Refresh the page
        driver.refresh()
        time.sleep(5)

        # Step 4: Verify if login was successful
        if "Sign In" not in driver.page_source:
            print("✅ Successfully logged in using cookies!")
        else:
            print("⚠️ Failed to log in using cookies. Attempting full login...")

    except Exception as e:
        print(f"❌ Error: {e}")


# Replace these with your Glassdoor account email and password
username = 'replace with your username'
password = 'replace with your password'

# First-time login to save cookies
login_and_save_cookies(username, password)

Login successful!
✅ Login successful! Cookies saved.


In [27]:
#login with cookies, still with problems
'''
options = uc.ChromeOptions()
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/132.0.6834.111 Safari/537.36")

driver = uc.Chrome(options=options)

# Example: Directly log in using saved cookies
load_cookies_and_access()

# Keep the browser window open
print("Login complete. You can continue performing other actions.")
'''

Cookies loaded successfully.
Failed to log in using cookies. Please check cookies validity.
Login complete. You can continue performing other actions.


In [12]:
def judgecommenttype(source):
    """
    Determines the recommendation status based on the provided HTML element.

    Args:
        source: A BeautifulSoup element representing the recommendation div.

    Returns:
        str: "No Data" if no information is available,
             "Yes" if the recommendation is positive,
             "No" if the recommendation is negative,
             "Unknown" if the status cannot be determined.
    """
    if source is None:
        return None
    
    # Get the list of classes from the div element
    class_list = source.get("class", [])

    # Determine the recommendation status
    if any("rating-icon_noDataStyles" in c for c in class_list):
        return "No Data"
    elif any("rating-icon_positiveStyles" in c for c in class_list):
        return "Yes"
    elif any("rating-icon_negativeStyles" in c for c in class_list):
        return "No"
    else:
        return "Unknown"



In [8]:
'''
# Extract all available "data-test" attributes from the HTML.
# This helps identify useful attributes for scraping.

for tag in soup.find_all(attrs={"data-test": True}):
    print(tag["data-test"])
'''


'\n# Extract all available "data-test" attributes from the HTML.\n# This helps identify useful attributes for scraping.\n\nfor tag in soup.find_all(attrs={"data-test": True}):\n    print(tag["data-test"])\n'

In [13]:
def process_source(gvkey, web, source):
    soup = BeautifulSoup(source, "html.parser")
    review_divs = soup.findAll("div", {"data-test": "review-details-container"})
    print(f"Found {len(review_divs)} reviews")

    outpageinfo = []

    for var in review_divs[:10]:  # Only process the first 10 reviews
        review_info = {}

        # Extract review ID
        review_info["review_id"] = var.get("id", "").replace("empReview_", "")

        # Extract review title
        review_info["title"] = var.find("h3", {"data-test": "review-details-title"}).text.strip() if var.find("h3", {"data-test": "review-details-title"}) else None

        # Extract Pros
        review_info["pros"] = var.find("span", {"data-test": "review-text-PROS"}).text.strip() if var.find("span", {"data-test": "review-text-PROS"}) else None

        # Extract Cons
        review_info["cons"] = var.find("span", {"data-test": "review-text-CONS"}).text.strip() if var.find("span", {"data-test": "review-text-CONS"}) else None

        # Extract Advice to Management
        advice_title = var.find("p", {"class": "review-text_textTitle__h8c3S"})
        review_info["advice"] = (
            advice_title.find_next_sibling("p", {"class": "review-text_isExpanded__wRVsY"}).text.strip()
            if advice_title and "Advice to Management" in advice_title.text else None
        )

        # Extract rating
        review_info["rating"] = var.find("span", {"data-test": "review-rating-label"}).text.strip() if var.find("span", {"data-test": "review-rating-label"}) else None

        # Parse recommendation status
        recommend_div = var.find("div", {"class": lambda x: x and "rating-icon_ratingContainer" in x})
        if recommend_div:
            class_list = recommend_div.get("class", [])
            review_info["recommend"] = (
                "No Data" if any("rating-icon_noDataStyles" in c for c in class_list) else
                "Yes" if any("rating-icon_positiveStyles" in c for c in class_list) else
                "No" if any("rating-icon_negativeStyles" in c for c in class_list) else "Unknown"
            )
        else:
            review_info["recommend"] = None

        # Parse CEO approval status
        ceo_approval_div = var.find("div", {"class": lambda x: x and "rating-icon_ratingContainer" in x})
        if ceo_approval_div:
            class_list = ceo_approval_div.get("class", [])
            review_info["ceo_approval"] = (
                "Yes" if any("rating-icon_positiveStyles" in c for c in class_list) else
                "No" if any("rating-icon_negativeStyles" in c for c in class_list) else "No Data"
            )
        else:
            review_info["ceo_approval"] = None

        # Parse business outlook
        business_outlook_div = var.find("div", {"class": lambda x: x and "rating-icon_ratingContainer" in x})
        if business_outlook_div:
            class_list = business_outlook_div.get("class", [])
            review_info["business_outlook"] = (
                "Positive" if any("rating-icon_positiveStyles" in c for c in class_list) else
                "Negative" if any("rating-icon_negativeStyles" in c for c in class_list) else "No Data"
            )
        else:
            review_info["business_outlook"] = None

        # Extract number of helpful votes
        helpful_button = var.find("button", {"data-test": "review-helpful-button"})
        review_info["helpful_count"] = ''.join(filter(str.isdigit, helpful_button.text.strip())) if helpful_button else "0"

        # Extract management response
        review_info["response"] = var.find("div", {"class": "review-response"}).text.strip() if var.find("div", {"class": "review-response"}) else None

        # Append extracted data to the final list
        outpageinfo.append(review_info)

    return outpageinfo


In [14]:
def processpages(gvkey, page_number, glassdoorweb):
    print(f"{gvkey} - Processing Page {page_number}: {glassdoorweb}")

    url = glassdoorweb.replace('.htm', f"_P{page_number}.htm")
    driver.get(url)

    # Step 1: Wait for the page to fully load
    timeout = 15  # Maximum wait time in seconds
    start_time = time.time()

    while True:
        if driver.execute_script("return document.readyState") == "complete":
            print(f"✅ Page {page_number} fully loaded!")
            break
        elif time.time() - start_time > timeout:
            print(f"⚠️ Timeout reached for Page {page_number}, but proceeding anyway.")
            break
        time.sleep(1)

    # Step 2: Click all "Expand" buttons for Advice to Management
    try:
        expand_buttons = driver.find_elements(By.CLASS_NAME, "expand-button_ExpandButton__Wevvg")
        expand_click_count = 0

        for button in expand_buttons:
            driver.execute_script("arguments[0].click();", button)  # Use JS click to avoid interception
            expand_click_count += 1
            time.sleep(0.5)  # Small delay to prevent too fast clicking

        print(f"✅ Clicked {expand_click_count} 'Expand' buttons.")
    except Exception as e:
        print(f"⚠️ No 'Expand' buttons found or error: {e}")

    # Step 3: Get page source **after clicking buttons**
    source = driver.page_source
    trans = process_source(gvkey, glassdoorweb, source)

    return trans

In [15]:
def processpages_2(gvkey, page_number, glassdoorweb):  # More like human actions with scrolling actions
    print(f"{gvkey} - Processing Page {page_number}: {glassdoorweb}")

    url = glassdoorweb.replace('.htm', f"_P{page_number}.htm")
    driver.get(url)

    # Step 1: Wait for the page to fully load
    timeout = 15  # Maximum wait time (seconds)
    start_time = time.time()

    while True:
        if driver.execute_script("return document.readyState") == "complete":
            print(f"✅ Page {page_number} fully loaded!")
            break
        elif time.time() - start_time > timeout:
            print(f"⚠️ Page {page_number} load timeout, proceeding anyway.")
            break
        time.sleep(1)

    # Step 2: Scroll and find "Expand" buttons
    scroll_attempts = 0
    max_scroll_attempts = 10  # Limit the maximum number of scroll attempts
    expand_click_count = 0

    while scroll_attempts < max_scroll_attempts:
        # Scroll down by 500px
        driver.execute_script("window.scrollBy(0, 500);")  
        time.sleep(1)  # Wait for loading

        # **Find currently visible 'Expand' buttons**
        expand_buttons = driver.find_elements(By.CLASS_NAME, "expand-button_ExpandButton__Wevvg")
        
        if expand_buttons:
            for button in expand_buttons:
                driver.execute_script("arguments[0].click();", button)  # **Click and then wait**
                expand_click_count += 1
                time.sleep(0.5)  # **Prevent clicking too fast**
            print(f"🔹 Found and clicked {len(expand_buttons)} 'Expand' buttons")
        else:
            print("✅ No new 'Expand' buttons found, stopping scrolling.")
            break  # **Stop scrolling if no new buttons are found**
        
        scroll_attempts += 1

    print(f"✅ Total {expand_click_count} 'Expand' buttons clicked")

    # Step 4: Parse page HTML content
    source = driver.page_source
    trans = process_source(gvkey, glassdoorweb, source)

    return trans


In [51]:
# Get the directory where the script is running
script_dir = os.getcwd()

# Define the folder path in the same directory as the script
folder_path = os.path.join(script_dir, "Review_202502")

def savefile(data, filename):
    """Saves data to a pickle file in the Review_202502 directory."""
    try:
        file_path = os.path.join(folder_path, f"{filename}.pkl")
        
        with open(file_path, 'wb') as f:
            pickle.dump(data, f)
        
        print(f"File saved successfully: {file_path}")
    
    except Exception as e:
        print(f"Error saving file {filename}: {e}")

In [12]:
'''
firm=russell3000[0]

gvkey = firm['gvkey']
conm = firm['conm']
glassdoorweb= firm['glassdoor_web'].replace("//Reviews/","/Reviews/")
print(0,gvkey,conm,glassdoorweb)
'''

0 GVKEY_001004 AAR CORP https://www.glassdoor.com/Reviews/AAR-Reviews-E4.htm


In [15]:
#print(processpages(gvkey, 2, glassdoorweb)) # review info on the page 2 for firm with the gvkey

GVKEY_001004 - Processing Page 2: https://www.glassdoor.com/Reviews/AAR-Reviews-E4.htm
✅ Page 2 fully loaded!
✅ Clicked 9 'Expand' buttons.
Found 10 reviews
[{'review_id': '91691291', 'title': 'AAR work culture', 'pros': 'unlimited time off it seems like.', 'cons': 'the pay is low for this type of profession.', 'advice': None, 'rating': '2.0', 'recommend': 'No', 'ceo_approval': 'No', 'business_outlook': 'Negative', 'helpful_count': '0', 'response': None}, {'review_id': '91570526', 'title': 'Aviation Services', 'pros': 'Camaraderie, flying, aircraft, location, purpose,', 'cons': 'Relatively poor salary, disjointed, amateurish, weak leadership, myopic leadership, no loyalty to hard working employees, no upward mobility.', 'advice': None, 'rating': '2.0', 'recommend': 'No', 'ceo_approval': 'No', 'business_outlook': 'Negative', 'helpful_count': '0', 'response': None}, {'review_id': '91417654', 'title': 'Sales Manager', 'pros': 'Work from Home\nViable Product', 'cons': 'Production issues\nP

In [52]:
list_of_f = range(800, 801)  # Process firms in this range

for f in list_of_f:
    firm = russell3000[f]

    # Check if the firm has a valid Glassdoor link and data isn't already scraped
    if isinstance(firm['glassdoor_web'], str) and len(firm['glassdoor_web']) > 1 and \
       "{}.pkl".format(firm['gvkey']) not in os.listdir("Review_202502"):
        
        gvkey = firm['gvkey']
        conm = firm['conm']
        glassdoorweb = firm['glassdoor_web'].replace("//Reviews/", "/Reviews/")

        print(f"Processing {f}: {gvkey} - {conm} - {glassdoorweb}")

        # Step 1: Open the Glassdoor page
        driver.get(glassdoorweb)

        # Step 2: Wait for page to load completely
        timeout = 15
        start_time = time.time()

        while True:
            if driver.execute_script("return document.readyState") == "complete":
                print("✅ Page fully loaded!")
                break
            elif time.time() - start_time > timeout:
                print("Timeout reached, proceeding with scraping.")
                break
            time.sleep(1)

        # Step 3: Extract total number of reviews
        try:
            # Wait until the review count element appears
            review_count_element = WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.XPATH, "//body//*[contains(text(), 'Viewing') and contains(text(), 'Reviews')]"))
            )
            result_text = review_count_element.text.strip()
            print(f"✅ Found review count text: {result_text}")

            # Extract total reviews using regex
            match = re.search(r"of\s([\d,]+)\sReviews", result_text)
            if match:
                total_reviews = int(match.group(1).replace(",", ""))
                max_page = (total_reviews + 9) // 10  # Ensure we capture all pages
            else:
                total_reviews = 0
                max_page = 1  # Default to 1 page if count is missing

        except Exception as e:
            print(f"⚠️ Error extracting review count: {e}")
            total_reviews = 0
            max_page = 1  # Safe fallback

        print(f"Total Reviews: {total_reviews}, Max Pages: {max_page}")
        print("*" * 100)

        # Step 4: Scrape individual review pages if available
        try:
            if total_reviews > 0:
                output = []
                i = 1

                while i < max_page + 1:  # Limit to first 3 pages for testing
                    temp = []
                    print(f"Scraping page {i} of {max_page} (Collected: {len(output)} reviews)")

                    try:
                        # Random pre-delay before page scrape
                        pre_delay = random.uniform(2, 8)
                        print(f"⏳ Waiting {round(pre_delay, 2)} seconds before scraping page {i}...")
                        time.sleep(pre_delay)

                        temp = processpages(gvkey, i, glassdoorweb)

                        # Random post-delay after page scrape
                        post_delay = random.uniform(3, 10)
                        print(f"🛑 Waiting {round(post_delay, 2)} seconds after scraping page {i}...")
                        time.sleep(post_delay)

                        if temp:
                            output.extend(temp)
                            i += 1
                    except Exception as e:
                        print(f"⚠️ Error on {conm}, page {i}: {e}. Retrying with extended delay...")

                        # Introduce a longer random delay before retrying
                        retry_delay = random.uniform(7, 15)
                        print(f"Retrying after {round(retry_delay, 2)} seconds...")
                        time.sleep(retry_delay)

                    print(f"Total reviews collected: {len(output)}")

                print(f"✅ Successfully collected {len(output)} reviews for {conm}")
                savefile(output, gvkey)  # Save the scraped data

        except Exception as e:
            print(f"❌ No Review Content for {conm}: {e}")


Processing 800: GVKEY_009664 - SHENANDOAH TELECOMMUN CO - https://www.glassdoor.com/Reviews/Shentel-Reviews-E6931.htm
✅ Page fully loaded!
✅ Found review count text: Viewing 1 - 10 of 175 Reviews
Total Reviews: 175, Max Pages: 18
****************************************************************************************************
Scraping page 1 of 18 (Collected: 0 reviews)
⏳ Waiting 7.71 seconds before scraping page 1...
GVKEY_009664 - Processing Page 1: https://www.glassdoor.com/Reviews/Shentel-Reviews-E6931.htm
✅ Page 1 fully loaded!
✅ Clicked 9 'Expand' buttons.
Found 10 reviews
🛑 Waiting 4.99 seconds after scraping page 1...
Total reviews collected: 10
Scraping page 2 of 18 (Collected: 10 reviews)
⏳ Waiting 7.11 seconds before scraping page 2...
GVKEY_009664 - Processing Page 2: https://www.glassdoor.com/Reviews/Shentel-Reviews-E6931.htm
✅ Page 2 fully loaded!
✅ Clicked 9 'Expand' buttons.
Found 10 reviews
🛑 Waiting 6.62 seconds after scraping page 2...
Total reviews collected: 20


In [53]:
#To print the data of .pkl file
file_path = "Review_202502\GVKEY_009664.pkl"  # Update with the actual path

with open(file_path, "rb") as f:
    data = pickle.load(f)

print(data)  # This will display the content

[{'review_id': '94109782', 'title': 'Great company', 'pros': 'Feels more like a small time business', 'cons': 'unrealistic annual goals every year employed', 'advice': None, 'rating': '5.0', 'recommend': 'Yes', 'ceo_approval': 'Yes', 'business_outlook': 'Positive', 'helpful_count': '0', 'response': None}, {'review_id': '92246112', 'title': 'Transition Pains', 'pros': 'Good local people trying to provide a service to their community', 'cons': 'Growing pains due to recent merger', 'advice': None, 'rating': '3.0', 'recommend': 'No Data', 'ceo_approval': 'No Data', 'business_outlook': 'No Data', 'helpful_count': '0', 'response': None}, {'review_id': '92152608', 'title': 'Great job for family', 'pros': 'Family first \nGood pay \nWould recommend', 'cons': 'On call pay is not hood', 'advice': None, 'rating': '5.0', 'recommend': 'No Data', 'ceo_approval': 'No Data', 'business_outlook': 'No Data', 'helpful_count': '0', 'response': None}, {'review_id': '92116485', 'title': 'Great company', 'pros

<>:1: SyntaxWarning: invalid escape sequence '\G'
<>:1: SyntaxWarning: invalid escape sequence '\G'
C:\Users\xxiao4\AppData\Local\Temp\ipykernel_53736\2224940158.py:1: SyntaxWarning: invalid escape sequence '\G'
  file_path = "Review_202502\GVKEY_009664.pkl"  # Update with the actual path


In [54]:
# Close the browser when done
driver.quit()
print("Session ended, browser closed.")

Session ended, browser closed.
